In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from dataclasses import dataclass
from typing import Optional, Iterable, Tuple, Dict, Any
from apache_beam.metrics.metric import Metrics

GOOD = beam.pvalue.TaggedOutput
BAD_TAG = 'bad'

exit()

@dataclass
class Record:
    user_id: str
    amount: float
    currency: str
    event_time: str  # parse to datetime if you like

class ParseCSV(beam.DoFn):
    bad_rows = Metrics.counter('dq', 'bad_rows_parse')
    def process(self, line: str):
        try:
            user_id, amount, currency, event_time = [x.strip() for x in line.split(',')]
            yield Record(user_id=user_id, amount=float(amount), currency=currency, event_time=event_time)
        except Exception as e:
            self.bad_rows.inc()
            yield beam.pvalue.TaggedOutput(BAD_TAG, {'line': line, 'reason': f'parse_error:{e}'})

class Validate(beam.DoFn):
    nulls = Metrics.counter('dq', 'null_required')
    out_of_range = Metrics.counter('dq', 'amount_out_of_range')
    bad_enum = Metrics.counter('dq', 'bad_currency')
    def process(self, r: Record, valid_currencies: set):
        # requireds
        if not r.user_id or r.amount is None or not r.currency:
            self.nulls.inc()
            yield beam.pvalue.TaggedOutput(BAD_TAG, {'record': r.__dict__, 'reason': 'null_required'})
            return
        # range
        if not (0.0 <= r.amount <= 1_000_000.0):
            self.out_of_range.inc()
            yield beam.pvalue.TaggedOutput(BAD_TAG, {'record': r.__dict__, 'reason': 'amount_out_of_range'})
            return
        # enum
        if r.currency not in valid_currencies:
            self.bad_enum.inc()
            yield beam.pvalue.TaggedOutput(BAD_TAG, {'record': r.__dict__, 'reason': 'bad_currency'})
            return
        yield r  # GOOD

class KeyBy(beam.DoFn):
    def process(self, r: Record):
        yield (r.user_id, 1)

class BuildDQSummary(beam.CombineFn):
    # simple aggregator; extend with more stats as needed
    def create_accumulator(self):
        return {'rows': 0}
    def add_input(self, acc, _):
        acc['rows'] += 1
        return acc
    def merge_accumulators(self, accs):
        out = {'rows': 0}
        for a in accs: out['rows'] += a['rows']
        return out
    def extract_output(self, acc):
        return acc

def run(input_paths: Iterable[str], output_good: str, output_bad: str, dq_summary_out: str, currencies=('EUR','USD','GBP')):
    opts = PipelineOptions(save_main_session=True, streaming=False)
    valid_cur = set(currencies)

    with beam.Pipeline(options=opts) as p:
        lines = (p
                 | 'CreatePaths' >> beam.Create(list(input_paths))
                 | 'ReadAll' >> beam.FlatMap(lambda path: open(path).read().splitlines()))
        parsed = (lines | 'Parse' >> beam.ParDo(ParseCSV()).with_outputs(BAD_TAG, main='good'))
        validated = (parsed.good
                     | 'Validate' >> beam.ParDo(Validate(), valid_currencies=valid_cur).with_outputs(BAD_TAG, main='good'))
        bad = (parsed.bad, validated.bad) | 'FlattenBad' >> beam.Flatten()

        # Good sink (e.g., to Parquet/BigQuery; using text here for brevity)
        _ = (validated.good
             | 'SerializeGood' >> beam.Map(lambda r: f'{r.user_id},{r.amount},{r.currency},{r.event_time}')
             | 'WriteGood' >> beam.io.WriteToText(output_good, file_name_suffix='.csv', num_shards=1))

        # Dead-letter sink with reasons
        _ = (bad
             | 'PrettyBad' >> beam.Map(lambda d: str(d))
             | 'WriteBad' >> beam.io.WriteToText(output_bad, file_name_suffix='.txt', num_shards=1))

        # DQ summary (counts, distinct users, duplicates)
        row_count = (validated.good | 'CountRows' >> beam.combiners.Count.Globally())
        distinct_users = (validated.good
                          | 'UserKeys' >> beam.Map(lambda r: r.user_id)
                          | 'ApproxUniqueUsers' >> beam.combiners.ApproximateUnique.Globally(1024))
        duplicates = (validated.good
                      | 'KeyByUser' >> beam.ParDo(KeyBy())
                      | 'CountPerUser' >> beam.CombinePerKey(sum)
                      | 'OnlyDupes' >> beam.Filter(lambda kv: kv[1] > 1)
                      | 'CountDupes' >> beam.combiners.Count.Globally())

        _ = (
            {
                'rows': row_count,
                'distinct_users': distinct_users,
                'duplicate_user_ids': duplicates
            }
            | 'ToSummary' >> beam.CoGroupByKey()  # trick to wait for all; or use Flatten + CombineFn
        )

        # Simpler: write each metric as its own file
        _ = (row_count | 'WriteRowCount' >> beam.io.WriteToText(dq_summary_out + '/rows'))
        _ = (distinct_users | 'WriteDistinctUsers' >> beam.io.WriteToText(dq_summary_out + '/distinct_users'))
        _ = (duplicates | 'WriteDupes' >> beam.io.WriteToText(dq_summary_out + '/duplicate_user_ids'))


Failed to import GCSFileSystem; loading of this filesystem will be skipped. Error details: cannot import name 'storage' from 'google.cloud' (unknown location)


: 